# Lab 17: Retrieval evaluation and the isolation test

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [13]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [14]:
import numpy as np, shutil
from src import sample_lsa
recs = sd.records("deal_a") + sd.records("deal_b")
chunks = [{"chunk_id": f'{r["doc_id"]}:{r["block_id"]}', "deal_id": r["deal_id"], "doc_id": r["doc_id"], "tier": r["tier"],
           "block_ids": [r["block_id"]], "text": r["text"]} for r in recs]
vecs = llm.embed([c["text"] for c in chunks], sample_lsa.MODEL, compute=sample_lsa.encode).tolist()
embed = lambda text: llm.embed([text], sample_lsa.MODEL, compute=sample_lsa.encode)[0].tolist()
import chromadb                    # rerun-safe: drop the collection through the client, not the folder
try:
    chromadb.PersistentClient(path="data/index").delete_collection("lantern")
except Exception:
    pass                           # first run: nothing to delete

In [15]:
# From the slide: A vector store with scope enforced in the store
import chromadb, json

client = chromadb.PersistentClient(path="data/index")
col = client.get_or_create_collection(
    "lantern", configuration={"hnsw": {"space": "cosine"}})

col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[{"deal_id": c["deal_id"],
                    "doc_id": c["doc_id"],
                    "tier": c["tier"],
                    "block_ids": json.dumps(c["block_ids"])}
                   for c in chunks])

def search(qvec, scope: str, k: int = 10):
    if not scope:
        raise ValueError("scope is required")
    return col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

In [16]:
# From the slide: BM25 plus dense, fused with reciprocal rank fusion
from rank_bm25 import BM25Okapi
tok = lambda s: s.lower().split()
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def rrf(*rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = (score.get(doc, 0)
                          + 1 / (k + rank))
    return sorted(score, key=score.get, reverse=True)

def hybrid(query, qvec, scope, k=10):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query))
    ok = (scope, "public")         # scope BM25 too
    lex = [chunks[i]["chunk_id"]
           for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex)[:k]

In [17]:
# From the slide: The metrics, implemented and tested
import math

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)

def mrr(ranked, rel):
    for i, d in enumerate(ranked, 1):
        if d in rel:
            return 1 / i
    return 0.0

def ndcg_at_k(ranked, grades, k):
    gain = lambda g, i: (2**g - 1) / math.log2(i + 1)
    dcg = sum(gain(grades.get(d, 0), i)
              for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(gain(g, i)
               for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

In [18]:
import pandas as pd
qrels = [json.loads(l) for l in open("data/eval/qrels.jsonl")]
rows = []
for q in qrels:
    rel, grades = list(q["relevant"]), q["relevant"]
    dense = search(embed(q["query"]), q["scope"], k=20)["ids"][0]
    hyb = hybrid(q["query"], embed(q["query"]), q["scope"], k=20)
    for name, ranked in (("dense", dense), ("hybrid", hyb)):
        rows.append({"retriever": name, "category": q["category"], "recall@5": recall_at_k(ranked, rel, 5),
                     "mrr": mrr(ranked, rel), "ndcg@10": ndcg_at_k(ranked, grades, 10)})
pd.DataFrame(rows).groupby("retriever")[["recall@5", "mrr", "ndcg@10"]].mean().round(3)

,recall@5,mrr,ndcg@10
retriever,,,
dense,1.000,1.000,0.958
hybrid,0.917,0.903,0.890


The isolation test from the slide runs unchanged: in this repository `data/eval/qrels.jsonl` and `data/dataroom/canaries_visible.json` are where Case Study 2 puts them.

In [19]:
chunk_meta = {c["chunk_id"]: c for c in chunks}.get

In [20]:
# From the slide: The isolation test, and proving it has teeth
import json, os, pytest
ROOT = os.environ.get("LANTERN_ISOLATION_FIXTURE", "data")
QRELS = f"{ROOT}/eval/qrels.jsonl"
CANS = f"{ROOT}/dataroom/canaries_visible.json"
QUERIES = [json.loads(l) for l in open(QRELS)]
CANARIES_B = json.load(open(CANS))["deal_b"]

@pytest.mark.parametrize("q", QUERIES)
def test_deal_a_never_sees_deal_b(q):
    hits = hybrid(q["query"], embed(q["query"]),
                  scope="deal_a", k=20)
    for cid in hits:
        meta = chunk_meta(cid)
        assert meta["deal_id"] in ("deal_a", "public")
        text = meta["text"]
        assert not any(c in text for c in CANARIES_B)

In [21]:
for q in QUERIES: test_deal_a_never_sees_deal_b(q)
print("isolation test passes on", len(QUERIES), "queries")
# Teeth: remove the scope filter and watch it fail
_safe = hybrid
def hybrid(query, qvec, scope, k=10):
    return [c["chunk_id"] for c in chunks][:k] + [c["chunk_id"] for c in chunks if c["deal_id"] == "deal_b"]
try:
    for q in QUERIES: test_deal_a_never_sees_deal_b(q)
    print("NO FAILURE: the test has no teeth")
except AssertionError:
    print("with the filter removed, the test fails: it has teeth")
hybrid = _safe

isolation test passes on 12 queries
with the filter removed, the test fails: it has teeth


**Exercise 17.** Write three more qrels for the transcript before looking at results. Which retriever wins on `cross_document` queries?

In [22]:
def hybrid_k(query, qvec, scope, k=10, rrf_k=60):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query)); ok = (scope, "public")
    lex = [chunks[i]["chunk_id"] for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex, k=rrf_k)[:k]

In [23]:
rows = []
for q in qrels:
    rel, grades, qv = list(q["relevant"]), q["relevant"], embed(q["query"])
    s = bm25.get_scores(tok(q["query"])); ok = (q["scope"], "public")
    bm = [chunks[i]["chunk_id"] for i in s.argsort()[::-1] if chunks[i]["deal_id"] in ok][:20]
    for name, ranked in (("bm25", bm), ("dense", search(qv, q["scope"], k=20)["ids"][0]),
                         ("hybrid k=10", hybrid_k(q["query"], qv, q["scope"], 20, 10)),
                         ("hybrid k=60", hybrid_k(q["query"], qv, q["scope"], 20, 60))):
        rows.append({"retriever": name, "recall@5": recall_at_k(ranked, rel, 5),
                     "mrr": mrr(ranked, rel), "ndcg@10": ndcg_at_k(ranked, grades, 10)})
changed = sum(hybrid_k(q["query"], embed(q["query"]), q["scope"], 10, 10)
              != hybrid_k(q["query"], embed(q["query"]), q["scope"], 10, 60) for q in qrels)
print("top-10 changes for", changed, "of", len(qrels), "queries")
pd.DataFrame(rows).groupby("retriever")[["recall@5", "mrr", "ndcg@10"]].mean().round(3)

top-10 changes for 4 of 12 queries


,recall@5,mrr,ndcg@10
retriever,,,
bm25,0.875,0.873,0.850
dense,1.000,1.000,0.958
hybrid k=10,0.958,0.903,0.895
hybrid k=60,0.917,0.903,0.890


### Exercise 16: answer (does rrf k change the ranking?)

| retriever | recall@5 | MRR | nDCG@10 |
|---|---|---|---|
| bm25 | 0.875 | 0.873 | 0.850 |
| dense | 1.000 | 1.000 | 0.958 |
| hybrid k=10 | 0.958 | 0.903 | 0.895 |
| hybrid k=60 | 0.917 | 0.903 | 0.890 |

Top-10 lists changed for 4 of 12 queries between k = 10 and k = 60.

**Does k change the ranking?** Yes, but modestly. A smaller k makes the top positions of each list
count more (1/(10+1) vs 1/(10+5) is a bigger difference than 1/(60+1) vs 1/(60+5)), so k = 10 improves
recall@5 (0.958 vs 0.917) and nDCG slightly; MRR is unchanged.

**Why hybrid loses to dense here:** the kit's lsa-sample-v1 embedding model was fitted on this very
corpus, so dense retrieval is unrealistically strong, and the naive whitespace tokenizer handicaps BM25.
On a real corpus with a general-purpose model and exact line-item names, BM25 should help on
table-seeking queries.

**Difference from the handout:** recall@5, MRR and dense match the handout exactly; nDCG@10 and the
"changed" count differ (handout: 0.870 / 0.906 / 0.897 and 8 of 12). For these short queries most chunks
score exactly 0 in BM25, so their order in the BM25 list comes from an unstable sort, which shifts lower
positions (nDCG, the top-10 comparison) between machines. Dropping zero-score BM25 hits and breaking ties
by chunk_id would make this reproducible.

In [24]:
# Exercise 17: three new qrels (written before running), scored in record mode for new query strings
llm.MODE = "record"   # lsa-sample-v1 embeds new queries locally (no key); writes them to the embed cache

new = [
    {"qid": "w-13", "category": "numeric_lookup", "scope": "deal_a",
     "query": "What is Wren's trailing twelve-month net revenue retention?",
     "relevant": {"DR-deal_a-002:t003": 2, "DR-deal_a-001:p0003_b002": 1}},
    {"qid": "w-14", "category": "cross_document", "scope": "deal_a",
     "query": "Which customer contracts require consent on a change of control?",
     "relevant": {"DR-deal_a-002:t004": 2, "DR-deal_a-003:p0001_b003": 2}},
    {"qid": "w-15", "category": "data_room_claim", "scope": "deal_a",
     "query": "What does management say drove growth this year?",
     "relevant": {"DR-deal_a-002:t001": 2, "DR-deal_a-001:p0002_b001": 1}},
]
qrels = qrels + new

rows = []
for q in qrels:
    grades, qv = q["relevant"], embed(q["query"])
    s = bm25.get_scores(tok(q["query"])); ok = (q["scope"], "public")
    bm = [chunks[i]["chunk_id"] for i in s.argsort()[::-1] if chunks[i]["deal_id"] in ok][:20]
    for name, ranked in (("dense", search(qv, q["scope"], k=20)["ids"][0]),
                         ("hybrid", hybrid(q["query"], qv, q["scope"], k=20)),
                         ("bm25", bm)):
        rows.append({"retriever": name, "category": q["category"], "ndcg@10": ndcg_at_k(ranked, grades, 10)})
llm.MODE = "replay"   # back to replay for the rest of the notebook

print(len(qrels), "qrels")
pd.DataFrame(rows).pivot_table(index="category", columns="retriever", values="ndcg@10").round(2)

15 qrels


retriever,bm25,dense,hybrid
category,,,
cross_document,0.85,0.98,0.91
data_room_claim,0.93,0.93,0.94
definitional,1.00,1.00,1.00
numeric_lookup,0.83,0.94,0.88
section_seeking,1.00,1.00,1.00
table_seeking,0.50,1.00,0.63


### Exercise 17: answer

**The three qrels** (written from the transcript before running anything):
- `w-13` numeric_lookup: "What is Wren's trailing twelve-month net revenue retention?" →
  transcript `t003` (grade 2), CIM NRR claim `p0003_b002` (grade 1).
- `w-14` cross_document: "Which customer contracts require consent on a change of control?" →
  transcript `t004` and the scanned contract's change-of-control clause `DR-deal_a-003:p0001_b003` (both grade 2).
- `w-15` data_room_claim: "What does management say drove growth this year?" →
  transcript `t001` (grade 2), CIM growth claim `p0002_b001` (grade 1).

They were embedded in record mode: `lsa-sample-v1` computes new query vectors locally, with no API key.

**nDCG@10 by category (15 qrels):**

| category | bm25 | dense | hybrid |
|---|---|---|---|
| cross_document | 0.85 | **0.98** | 0.91 |
| data_room_claim | 0.93 | 0.93 | **0.94** |
| numeric_lookup | 0.83 | **0.94** | 0.88 |
| table_seeking | 0.50 | **1.00** | 0.63 |

**Which retriever wins on cross_document?** Dense, at 0.98, ahead of hybrid (0.91) and BM25 (0.85).
That agrees with the sample (dense 0.97). With my new query included, dense's lead over BM25 grew
(BM25 fell from 0.91 to 0.85). Cross-document questions need passages that say the same thing in
different words: the transcript says "require consent on a change of control", while the contract
says "may terminate ... if Wren undergoes a change of control without consent". Dense matching on
meaning handles that; BM25 only scores shared words.

**Caveat:** dense is unrealistically strong in the kit because lsa-sample-v1 was fitted on this
same small corpus, and BM25 uses a naive whitespace tokenizer. On a real corpus I'd expect BM25 to
do better on exact terms, line items and tickers.